<div class="alert alert-info" markdown="1">

#### Homework 7, Problem 6 Supplemental Notebook

# Linear Systems

### Math 124: Vectors, Matrices, and Applications, Fall 2026

<small><a style="text-decoration: none" href="https://math124.org">math124.org</a></small>

</div>

### Instructions

Most homeworks and some labs will have a Jupyter Notebook, like this one, containing Python code that supplements our understanding of the relevant mathematical ideas of the week.

---

Open this notebook in Google Colab. Instructions on how to use Google Colab are at [math124.org/running-code](https://math124.org/running-code).

**Most importantly, make sure to click "Copy to Drive" and rename the notebook before proceeding!**

---

This notebook contains **Homework 7, Problem 6 (18 points)**. You won't need to submit this notebook anywhere. Follow the orange submission instructions below and include the requested screenshots and written explanations in your homework PDF for Pensive.

Work from top to bottom. Most of the code is provided; replace the `...` in the marked cells. You can add extra cells to experiment.

## Problem 6: The Score Adds Up 🏀

---

The [WNBA Finals just concluded](https://www.espn.com/wnba/story/_/id/50343701/golden-state-valkyries-win-game-7-secure-first-wnba-title), and the Golden State Valkyries were crowned champions for the first time in their history! Billy has been following along, and he's gotten curious about the numbers behind the scores.

Suppose we give Billy a spreadsheet containing shooting statistics and final scores from hundreds of games. **Could he work backward from the data to figure out how scoring works?** Would looking at more games help? What about collecting more statistics?

We'll investigate these questions using real data from the **2026 WNBA regular season**. Each row describes one team's performance in one game. That means each game appears twice, once for each team; there are 660 rows from 330 games.

Don't worry: you don't need to know anything about basketball to complete this problem.

## Setup

---

Run the next cell to import the libraries we will use.

In [ ]:
import numpy as np
from scipy.linalg import lu, solve_triangular
import pandas as pd
from IPython.display import display
import base64
import io
import zlib

np.set_printoptions(precision=4, suppress=True)

## Meet the data

---

Run the next cell without editing it. A fixed copy of the data (originally downloaded from [here](https://github.com/sportsdataverse/wehoop-wnba-data)) is included in this notebook, so you don't need to download or upload another file.

Here are the four statistics we'll work with:

| Column | Meaning |
|---|---|
| `FGM` | Field goals made: successful baskets from the floor, including **both** two-pointers and three-pointers. |
| `3PM` | Three-pointers made: the successful baskets shot from beyond the three-point line. These are **already included in `FGM`**. |
| `FTM` | Free throws made: successful shots taken from the free-throw line. These are **not included in `FGM`**. |
| `PTS` | The team's final score. |

In [ ]:
#@title Load the WNBA box scores { display-mode: "form" }
_wnba_data = """eNqVXduKJjeSvt+nqAfIgdRZuhxm8LBgL4Y2M+zVUmMXduN2lemu2Z1++00d/owvQkopDU3TNvriz1SG4hyhn59/e/mfjz9tPz2/
v2zvL8+/bW+///72+vL6vn3zt+828/132zc/fLd9/8OH/7C7is7HtG961/5Pu/vTHre/vL2+vvz4/vHHf70/ffjX6/ZfL//39N9v
n399+vbjP18+v3/djN7sFrbgxvhuvSRo3KYOhD/+9icJhSR+ePv89vr+9vTDy2+/v23/eP7yy8fXn9/fXp+++/rloPNlU2FzmYJ3
YwIDCKep3UFAmc3Hk4BGAn97+/TTy+vTh/djE5/+/vzp16+fP7582T68PL+/f3o5/v/b5982ozZ1ULFbUmMqfPkFzeM10qbSFveT
iCEiafvr86dPz1+e/nG8z5ftP19/+vj8+vz0zcv/vnzezAHUeTPUHsZgvp6RsvvxEQvWnliL2G+PpX9/+fn4+88/Hs/5/S9vL68f
//303cvnH//1+eum08EGdvN+jJbLBTUTN6Xy06d04h3i//LLxx+ff357+vDr1+37t8/vn55ff3r65uPnl80cn63sWhwj+Wqkczzy
sWN+i+ZEJkT+OePen5/++jkfm+8+Hnz75e3472+/vv47f+2waY0fm4HFck7LFH5zW6KP7E/wcRrkGeGcczx3LM+tx2i+WtLSoZw3
t0Xa6oDw7sQOjs+x6cfbGtx0RmMA6QWHzad23xJtf0QagkW+fTv+8frzy6eXL08ffn/+/OvBsybz7MHyu7sg0WME1coDx7d8vEja
d6RwcUolOx/7cYiP45skN6YjAdenP58Dv4V00iFRdrwq5yN2gqvkOKBhDGWLOR0dj6N7LAn6hGqECmbu3l5X2RHjGC7XC3LlwQ9m
ivaEG4R3nNMJgLz3YUt+jOfLR3yo6mmkxyfZdZDm50lojlBZMJgxlq8WCmMvH/s4ifTgDsHy6EreNVt+z82nMVws7xRv2uRre/br
ICqv2NUdsjfrzTQmcQFjQljXDfS0gQFJcJU1ONCHIK5SPIwJDCCc5sH8mVtRBIAQsfzc9FrAFW3tx1ixnJEyZfMsKYG0J8Su2L6y
bZbmFwRWfL/njTOkRpICkeVW3Gd8hSc7hq+4L7+9zrafOvEK8fwjjXSQL29/HP1dj0kMQMJiivkhNBg9SWn2Ej3z8BNtQuW+024R
BPjqMftmhALNoQz7DGj59EaXL+xnxli5nh28w1grj0/7D5IrsLWS6/e8bTtwPYeK5UipqIoI6k05RHK1JOVdMfMd6AkOFsuFyaWy
2ZPZnfbZI5pzRmfau8wnMY2xfLUQL67yGIkXFdhTL1ksWznZ1vJjAksWKzZOdg52ogBSKnKxNLb2qqJKekxggOGmSTb1ji0jPaES
4qVwEJKuqAiFgoqhhYUv7d2Yt2+HM6JBTvWeidj87N1mM38fo/nqzjDqTQtNEuqgxFmFv8jx4ClzD/w0w/LVgunCVsxRMku0Riwe
Sa6UdDXKidM5kC1mB7uogwiCUBv2i1OXpvxs/sT7GL1wabojpkkeHZt2YYZ0GvHY8axOItgSnFCHuDbjfbEK6XUc0hECUrBcLJLK
omLncL5eWtPFkmfGvPaIHkiIjm/Lac++WRjT6Nz5odSJRWbSRyGxpaeejLF1+0i/cOjckwkbChodEXnxvcTh8VXgkrzhRPjqKxZI
xc1mkiMhlYWPbw65cxjoaQxenIe4FU4mT9CQ0NJmptp1iaYYCucJ6Fy12+IFElQhlMsrwfPl9DPxwcFCznL9sFdDmoSH0YhdBxKy
nN6LC+kuSCwDCYcllY+sQYPWgAyzglG7I1edQdRyHC4Bgu9z+C1LLtoDi2h2ZEaugCoKmoxBDu8QXL0XfjskznlejUM059aBsaCL
F3lsOckbTmGAGZyXAwPMBwLLrZypQ6TFLDmDHcNXrlTxQQ+tSdsXEC4P6JXM0NWuJrXByVy509K1so9gFO0mCLAw2AzpV8TCzBZM
bU5BAnpV6poSgx1NSAElh5AFRfw6dMo4lC9nIqiEEjJWnV/C7oidxnR1alHoeIGeBnVLBD2BmW4Ve+GpjZu1dgmDqDF6YePmDEw2
9witEb0+gLrGgrwdE1ifv6x3DwJkglmQXsLFkKLTFF5JIH05WKzntoLP7iRYbNYi9JbCN+X076CqOZFbCr+GUxLYOxZEWLpx6PKb
5C10YwLrM6eq+PD0BB4JDJTY2NkrVMjX41QGkHFYo/oR9F0CUpmZIHtNxrk4hk5MkKqIHIQXbESoUMHSikjFfPFjLF8ttO/hpWW7
k6AksMw+DyBnYABjnUNn8eOSMy265/Ty3I7gLoI3sNJbBIwEPScxgAwSSrEw/il8nEISUx83Bwby8bNj7MzHNS1TCm6SI6FlLt0+
GVUqhk+OwqsxGbH+6vibvAsYTnMPGXb8UVxodfk8x3M+HMpXc8trr4kbd3KeswhdJG6sbUEx0JecwCp1o4uXg66Gc4SXXl5nqehm
NMOuMbgEiGPn89snkJjOMzSIhpHSy7x/PD6dPQ4fQIS/47nWcAHhV6lDGdtSxVcj3cep8OVXhmMOFWUDnL5CRCI3PKCsAhOyICew
TqTu1XEkz9slomB6bdUFOVXxHslm5Xi+fJTEy+qTnCe/I1qysTiANifSFDACR/PVA2M5bVA7krwisHT7RKTB1Pg92W0cy1eLgPb+
MFZP6e81gi94pY/ll1ATiGBORqyfhDsMT8N5EGI8jdGJgWy4sMiBwIr17BQWCejB8fUWsdzwHNQQ1LwAywFxCgOQtEBTifGeMtg7
xEt+ERowezuZBeIFmq3ueM8+3CxiXZBhXn49EXXNmtuDxcqxfPFA8itWM5V8QPRN7fso/CHJw8nc0746tBoA4OKIZPjZGZm9uoUg
1J7GJAagTh0Vm5eksE9IgJ/m3nxPVQS7MbpbL0TJwbPFCDmlQQAZNvO3qxDy8CE5cuJu57OXQyfnIweFyFGkWWq/YjRRtoET4KtH
5qvfymcn9g8gvO5VL5byERIfnMCyfLGkYLGaKgWDBJZp7BooVlh/wCkss9jF+8ibcAqgYJGAOEJCBtTyS4VxSw7n6we+u+YWXAD5
JV0uxj2qxhxIZ3MkrhUHzVaPg45K8IicVWzU4rkdN4thpxUb8VEjSdwSEHyzWqyV4BlQlpzOzWqx6rJiuCqQxBqUEQqTy9b8IpRP
cLg4+L3SjqUGg7LKISF8cFw7vycV8RHHBATjDYzW8v4WZF4kyXWoFeQg4TvmgFkpCfBjKF/O2DY77CVSeKqaqBDKebWXOKaZC/sY
3gGE2aezrC56n35fIwHGsl2aIZRqbZLYHCuXc2ezFSxReDoaBMsKYS7qC7S4624MF8K+qzbKypGkbCQZdZyAZYl4KdfNZqIbE1iW
iOd0VAmUEMM4JHCrRLwILo3ChxO5VyHumsVKSi+SCBuUDQrWz6FeD7KTgwXjy0O/N3EP+xgQLj6bdDZqeSuToBwvAMOaM0xsxIho
zvrczs7HNifIzBjKV3Ox37tIMbGXvpFRjy2jDpEWTuRGSt2WWE1CiZ1AatlbpdvN4saaRU7kTu12STgkSBckhSS4yOrSO6p4bPgW
DM3Xd/l9EelIIMDcPM9QD44HT42DZ3mGUnyGId5kECoDbYybrH6E2fYwhvP10mQpBkuEVFqyCL7ytIbRXpGl4ZQGkCshtGe72UGQ
PoEkk75OzwGm8qEZo/n6TnW3VpHzICWQYH2ibawNLA/5chKjyJ+0+301A8h4SYE9BZgMvQ3gHlazG6N7twMtEFPtJmDjiGDGt1PT
EQuuOI0LGD8S6VG6StuYkMiAnSRr5zOloQVHUJDqYCgTiwP1KBwJ+76zZ5i3LckCbglf9C3pUIugA8EVg7NzLYyB4ktoqqCWYBF2
lJ5zrXqn1wZRFqdRR5MTTsUHiWPwNOzosgkDZS4HGERZWjkdpVRPnRpcohc+RwmVRfbjFuG3FGB1uN3JdJLIrealvdaFAt84pHEv
+KpLmVtuP3JjMjeDr6FuaqRdIXF2SOoZM9i9HeJTGEv0lBtk780BDgieB95rUbamDKaELyLvrqV7CR0RfV3lamrQ9ZHjlsDrKlfz
KBSjnySBpdX9qKfDriFJ5WbKsVjT+BZqRyqCeS71oDmDr5LCANLbJqGw3inDlGJv0kvs7iwVEZ7OGmlJolt/UcDHSp4OIiTStJ6m
H1T2ibNzay6ws/RDbnqFaokDaxA7tYRtjduR/OfYpR2c5Q+9r0WsSLqPA1AovjhcArqOKYedhgfaIXqS67YlxH3wrR5DJ7luU1NM
2WzZTzBIqs75HJXp5XwHKWsO7xlN+A/Vhre46SCt7GLT2/NHyjBI/HzXa+DJkwd5wCPC1+e9lmr40/6XFG6cd1/tJdjDhBQmfdq+
JohBd3PopFHbN8eNnlvvCOVn4zpuUmolvB8TuYCJc1f8pwisr0FYuVvdPEW8gaTiFJbtPKUwsOykPq0GrZHE1AnOeSKo7JTYaa1d
Ttdkk/fkX20Q2x2grsmjJitjHOP58v74pur7w8+D0PJ3HL8iucKZ6pIUbvh9qhTr2M3R53PsIS4PgC3Zjiy/tB1jr09AMfeznURI
j8h7B0DU9ksidw+Aw3z1QQRkWFi0U7WuCAV+C4cv+qlqjezptR7oiOh1s3BVfAGFKCexbBc2prU5EAMkJMC3q+v4rh7fPsby1cLU
rjXZjgLoYTcgwOI87VUiPxbkFsdO016qpVnodxViOdsOzpB1j9i/NmMSA9AohFbCz+cRMBpJ4HHpezNYk66EylblzuAvfgrtnEHw
H2guCzxUwOnc7i1r/e6enuchw0JfcTf2f0s7JTwII7Csc5DdDgcBhwSkyJRBI1MsTzMGc9dL+v+mRa3o0T2CF96maSUrdPw4fOFt
1kpz2PdA4C5zMRwzkaO48OwM3wEGuZPqehIXRyQwOQDVSwv44gw6OwDZajq0JHlnZzNG6MvbrvshS9AIPjojckvzmPLtcu7pfJSz
O6NQmQbbymsYkH8cO4u11XIDf+bbDqwirDC3emuvdp2nMZYvl11VrTDk5JizKaNghbbrnJWSZiY7j4PFcuHg6erVkp10NmSESW3X
uLMrF+l6PSY0QFzQPhyVspeWNtIiJSkq5OkxtXSDTgCHy6aCvtihHj74jg7x62IH0WEjCayLHXSTH7SVILzCqsPPtZbaOEYvGvxE
ZduBDoiezqioOV97CZ4OqdhroI58NguC61L5CqPTN5eLxA+nwpdPe7oDWM82IZEFAxZ41nv7GL5gwCoMYOZG2N2O+LXpK4c4SApL
y9e2DpmzaOagAMIsrUzf4nzQAeLYqenbko3kcziN2EXcpSbbMEbH4YuwSx1MgA9uED2vesnIXHUQx+BpzUtxU4oMP3nGkdAa1Hdd
ca5tgQ87JnO3rZS3CBxUHFKZacHUSg4g48TBEzVoqvNAA3IOrGe7gHbPsLOhzsgJY/iow4mXe9Xij7Nc4SBA8kuppeGnK4UIuocT
WFp+teIG/C8XEb9KcabWzIvnlhFYJTmz7qRU/4FOiJ421YlmQomdNdX1hSJh9ySxRoPhOuOrDrFQY/i6paR6jafW8Arhy6Zas9dQ
EZ0YTmDZVFuT0wFsOK+RwHV+y9ehavDmDLiY4uIgnecNAucmZ0nG8QA3h8+NzlADK6RgPYgpI2dDDU56mSO4Y4SZUxhguuR2nWlG
DOeQwGoOYhGzWJspCSza6doYDOhpPAiAtLKrSUmulsOnC/SiizzV/SNJ6QOil0WWRVAkMPY5fl1judeyHDgwIKd4D1FfxV6MTFIS
HCoGpnCWtziG70AmRK6LWOQEQklhXcRiWl0oHbuwI4V7Xex7JUK2Bidyb2xNaPV15z4GkFl+HeurEd5wSWAZ6ctJobybjvYCZMhi
TluJkZTKsn2Mnk1qs6aOzYFQX7AIniZzy+GxGFjm4Fk21+yPYUHn0QkOwSsbJ7ThMcBADL8wcuocQfQtg2c/L6WOFHy+4uHnGb5z
8fqyYotzDA88SI646pyqs3eK5I9jAovmqZbZdXh2IuLn5Z0tHgp6h6On1Z2mRIXRzAgJwX8g1lInf9HMFUnqfrSlVBSls5k37BFE
SVo2VZb2CuhxkPhlS6WpndnEj1EhftpUVCKNtJsced1UVCMkDmymqBlycQZKzExhFQnHr85AzsXlM0RRxkiSS3fjsLk4qRkWh5YX
R4uscjcEu7bh0aNbBN9nwZqgQ0HCKd3nwFrgiKZMdEhp1dj6aGpzY/i0t9W4R6VJ8zySNolt5yLY5h+hljiGz6NtxQrVWNIaSZpq
fafdoJbJB/wOjMSNZgPjijFH8aYYkMJ8/1ULPl+A57u/11mQFPaOEcHzymrbKvvsGDwtrK7zn7QHKzImtvP83IwOgK1XUcC3YxRG
03O6LoVQ62ROzk87e/95e4+r/Y2kyTh63t9jy87n3m7K1CaF+EmLTQu0JVSEHDxrsgktU0J1MYnkp443y/rbDD07pnGnrr/080aw
xPDoxsXozjoEULOPx+DT0Z2tmtaC9koW0VPbt4BLqZQbo2e2r9GPsiQCOwTPw5ziOggJXoU5E4/OJ89++U4JaQkUMf3HidwpIrW1
Nc1iKXQCuZWmM0vVVruq9Bg6mVlaakgTpNNTROTdKyiK6lUOAtWczt0rKHKsz2DsKJEEM/v8ANg2TQgjIBw+n11bRBdkWtW+I3g6
RERvvJtCYmdVvKpqjVNoql0hdllYWs8ezCyUFJa1paWtQpHeUDtJLqOmqZVazWsgRy/Ai9yKKxlu+mGD2NVs8DpyDToYBH45Gjyb
vbkuVZ14i/jZ9LDq7pdBOHqMns4Pcy0uDtv2kFoxz76aVRSVvA5UUQrorKBC16jwKe7U2W5RoMs5gbUIJiGjMwLLOYF1cBUEG9TZ
dBFzGQzfJqmjUtOxpOIEXAC6GMsDTrseEb4eGlKCbUXJ+jGJ5dgQ49q8W7JU1NmBEdc1TPkLRPL0BHZVwlSOK33+s+UiTqaGDadl
USerpHIvo51aTeB59M/Gi5irgqaXMXRlkAI8vYyhzZ1ToK/V2XMRh9VQwtixj8FxboxflBLLq78OuEH4rKChWaoBmYejZxUN1dDC
0RMH2hLa3uSAqu3BThVU7nFAfLTz05uAHHOrmdUDNmTwxcxqMWf9QHtEr3hANnEJ+Kqa3JfBm5EePSB6eSeGbZkyPcav9F6ZF5mF
MP1+RPwq4utactKN4austm4O+imEFYgwv8jzqRrdI+XHwdMsX736Dko4ld4ZeJ1usW3qTRpTuJFuUa2M9Px2WiGF6aT6rn9DgGeT
6nuxo0FqrePs9bYxC8YWxy9jjKGO1iX7WhvEr2uHXQ30nxFmQWA9Iy3VKCvpHG2RwJ3eda1qNWvwYxp3etfb3CoPVpQG0RVXg0z2
Rz1gGMMXc0xqdBwsT+0RPRlcJCNrAjmZW+Qq058BVaUDImfVs/1dYxK96FVvzZnE9iCs0tK9ahNv1DknUxJYeVd2b9NFKTundEIK
8uRcZDYVTM6RJEaZzT5P1mZXnDTMjjSmd82FmhShD8ixsxaOvAGlaXM/P4FRCL4/Q8ToFuK3Y0r3Z4jUilZLZX3KkDRT+40pMhr7
MgR6mm91bQQAfAeD4Emlwd7qOOMYOSk0KN1zDhSHIbHVjRwbJjl99Vn8mMAqyekeSeJT+huH+BvRbd2uDPRjCuvodp11DV0wynj2
DDdGKek2yiuMSdzJbagWryQeJmkm66M6w3sKnVUy6nZvL7FOROidCzeLDEBByEncuHCzPkWi+J4yCUlMU931qp1EpYgCPM906zrD
7fxhS/Krq+8axOWPA280htg4fjErXNXXPlPayipE34kyV4GFsSpO49akAl1NONpBq9lzLA9hnfmrYX6JIHEjx6TKtGw6hBZkmJ3e
Ma7aGLUwhk5uGDftqmUyeqxF6NQAUa2Gm7QOB0/7d8LjNptT8FkQW249JVYOHJUE1pfct8lt8AQeCdy7mkPbNm1oH1O5eTVHTjfQ
TVEHkcAeZW74ylo3gV7YvbrN/yMVZEF++YnerTNuNO4gQ17r3epyeHzhhMjFfO7YslPuAj2dz923vCu3I3w6HT41g0GNsdPBPK1b
hQwcB6Lqpp/lm67exzRuzQgLj9sIaQNAWMU7d5SUCk3UOZzCnStKSk1hoBYk5QySmE6rC+1Kdgj3cfRqWp3mlqaz7P0Xc+ICd5w4
eD0mjkVanEPwPLzbGascPA3vVn+NHVYHgirdqCSOj9aXCwp3aolLPSZsXUACNwd01SIY78dEbo7nkr33ypHoksVEHfuUinoQAQw6
UbclVqTR0HAJoavL5VMb9hvH8NX13qFeBwuHxu+IX1bzW//wd86Of0FiWc7fpk1AMYzyJMHuTcgKbcaWHVNYT8wpMx+wiUN5jRTu
WP22tUCd88IEjRtmv3a1/5SEuTdIYhbzbI1omCni4GnMs0z883CSPcmwriBpNBzskTC8ILBye8vDF6OZnt4hgTvXZPnWQhfHJG5f
k+VACniPJBaplscnoAIBDl9eD9paYAlOkkyb2ciuvnFUYGf3U7kaLAKuiwidBDpdvdclXSCvA522XqmEEw2VT4hdNjHUCbVl1so+
JrFsYyjZqWL+nfI37EhhWs9b5T7mZzh4Vc7Lb5Q40CC0bt8MJIftCCo3lZ9qFjDp0KCRzLqta2+XaccxgbUa8O0y9FMVBZBei6uJ
9o1dDtdh51cTBdZPpYJF7KyUMja148fQSSFlrQZimjs4hl2Hug/mNTzDx0ncCXW7lh8lEiCv/B+6EjdzzwWZP3AlblEkJP9CQDqz
KxpNu/eARBHHLu5otHg1/YEFMRZuTw+qAQQPGpjTuTs9SKWau/X0Lgnp3MmDhNbWasYkbjCHbk9BoiWCeIvTluI2MxeahQR42lNc
G7w02qVRIXpZXZ1YY61AL25PSOesXcJrxM+Vgn7ck2vG6LlWKHIhwWeLhv30vdthyy17oJc4kduXw5bCEzIJI0i3NPeJuzGOAjyv
8A8PO+jUytEhej6+ruZtBfsw/Hx+XbmlnAWhI0kzs89vp27V4WjDcfT0dupekcWA4OUAm5r3Y2KQU1hOsDn4w7L2NBUjErh1YWYt
oYC+FEHk3n2ZD3UAHyIhlXn5n7htW4JX9X+1Qe00Cs7+jNT3mA4kSK1+pDwuhy+SIKYOUXLna5/9GUm2mY4z4GWQBYltDl9cmFr7
JHF+rzrbM1IXlBzJjdg+G2weI7DqK6x6R4MPdnZnpFGfahcKzuIba0YEfH5Xo7wu4YBb9vSze9pqZX3RHXYMn97T1nufZ4dGWkZA
/Vkta8boWQS0Nhdi+PbszEiD1tpRqWa9HFSN8cvpKbapfNKaZ2dGmjTYdmKnNjaQzuJU7skdXcvGSXedLRqp66vtLOq9bWQYg6d3
d9UoFMZTz7aMtOio9Q/Go7oDjp301Ip7VoLed0TeuzTLl/HwZwhB0Lh3Z5YqsxvPob16J8E1iIUOPGHHSg4FfjU9ztcB82Q16F0j
fjXIpq94FwQWo2xyrVbxH+j5DcLXl1T6dlFxGBNY3lFZ48E7HUK9WySwrvovtcYax98KEuvLQsVd9wcBx55hek9mbZCDHjeBnotf
mQLQOwkxvb4zUDejwY3hqzsDZQha7wHhc7HzCOBEP0bP7wyMtR820ouT0BrEXq9mdkfkXUbhTk9naNxLJBKSmDe86Fbw48fgabtL
i5yDxaHVzt7/8lre6izShegSeX0tb9NVkPrRSiH0xnXw9ZIt8DUkieWF8KXQJcKeK40E7mldX4vkT60liNxTuukxXOXUIAqEl1ke
vv0xw9WM8YvTp/c6hszT17AIn4XeO3NfYGfXZdhq8Z3WplYgr+zqnqZWaU4Sh6Pn9zSVZFP+ePTbnv323M8oQQYFQzwEfOFn1GAf
u2pCq4AEpgdetw6F00gT4OmBN/VWPBCWKiL4D9zVF+rwt3BB6Q/c1Rer7g30LROjNJ8faHW9smXfx/DF9EAI9vw/1wx58g=="""
games = pd.read_csv(io.StringIO(zlib.decompress(base64.b64decode(_wnba_data)).decode("utf-8")),
                    dtype={"game_id": str})
print(f"{len(games)} team-game records from {games['game_id'].nunique()} games")
print(f"Dates: {games['date'].min()} through {games['date'].max()}")
display(games.drop(columns="game_id").head(8))

## The scoring system

---

Suppose the scoring rule is

$$\text{score}=x_1(\text{FGM})+x_2(\text{3PM})+x_3(\text{FTM}).$$

The same coefficient vector $\vec x = \begin{bmatrix}x_1\\x_2\\x_3\end{bmatrix}$ must apply to each row, because (for example) a made three-pointer should always count for the same number of points. We can reason about what $\vec x$ must be, but let's see what the data tells us!

Run the next cell to create the matrix `A` and score vector `b`. The columns of `A` are ordered **FGM, 3PM, FTM**, and `b` has shape `(660, 1)`.

In [ ]:
A_with_col_names = games[["FGM", "3PM", "FTM"]]
A = A_with_col_names.to_numpy(dtype=float)
b = games[["PTS"]].to_numpy(dtype=float)
print("A has shape:", A.shape)
print("b has shape:", b.shape)

In [ ]:
A_with_col_names

## (P)LU with rectangular matrices

---

It's time to solve several systems of equations, which means it's time to consult our friend, the (P)LU factorization.

Recall, suppose we're given a matrix $A$ and vector $\vec b$, and need to solve for $\vec x$ in $A\vec x=\vec b$. If $PA = LU$, then to solve for $\vec x$,
- we first solve for $\vec y$ in $L\vec y=P\vec b$,
- then solve for $\vec x$ in $U\vec x=\vec y$.

We've defined a helper function below that takes in a matrix $A$ and vector $\vec b$ and returns four objects:
- the $P$, $L$, and $U$ such that $PA = LU$,
- **and** a vector $\vec y$ such that $L \vec y = P \vec b$, if one exists.

That is, it performs the first stage of the process of solving for $\vec x$ in $A \vec x = \vec b$. The reason it doesn't solve for $\vec x$ itself is that there may be one, infinitely many, or no solutions, and unless there is only one solution, Python would throw an error by default.

In [ ]:
def plu_system(A, b, display=False):
    # Don't worry about the implementation details below.
    from IPython.display import Math, display as show

    # Uses SciPy's lu function but rearranges so that PA = LU, as in class.
    Q, L, U = lu(A)
    P = Q.T
    k = U.shape[0]
    permuted_r = P @ b
    y = solve_triangular(L[:k, :], permuted_r[:k, :], lower=True)
    remaining = permuted_r - L @ y

    # Stop before back substitution if the full first-stage system fails.
    if not np.allclose(remaining, 0, atol=1e-8):
        raise ValueError(
            "No solution: L y = P b is inconsistent. "
            "The leading square part determines a candidate y, but it fails "
            "the remaining equations. Do not proceed to solve U x = y."
        )

    if display:
        def latex_matrix(M):
            rows = [" & ".join("0" if abs(value) < 1e-8 else f"{value:.4f}" 
                               for value in row) for row in M]
            return r"\begin{bmatrix}" + r" \\ ".join(rows) + r"\end{bmatrix}"

        show(Math(r"U = " + latex_matrix(U)))
        show(Math(r"\vec y = " + latex_matrix(y)))

    return P, L, U, y

Let's test out `plu_system` on the full dataset of scores. Passing the argument `display=True` renders both $U$ and $\vec y$, which is helpful in case we need to solve $U \vec x = \vec y$ by hand.

In [ ]:
P, L, U, y = plu_system(A, b, display=True)

To find the vector $\vec x$ such that $A \vec x = \vec b$, we have one more step left: solve for $\vec x$ in $U \vec x = \vec y$. To do this, we'll leverage `solve_triangular`, which we also saw last week:

In [ ]:
solve_triangular(U, y, lower=False)

## Task 1: Interpret the coefficients (1 point)

---

Does the answer for $\vec x$ above make sense in terms of how basketball scores are calculated, given that `FGM` includes three-pointers? Remember that

$$\text{score}=x_1(\text{FGM})+x_2(\text{3PM})+x_3(\text{FTM}).$$

If it helps, look at the first few rows of the dataset.

In [ ]:
games.iloc[:, 2:]

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include your explanation.</p>

## Task 2: Use the first three rows (1 point)

---

Use a (P)LU decomposition to solve the system formed by just the **first three rows** of `A` and `b` (indices 0, 1, and 2). Do so by filling in the `...`. _Hint: Pay close attention to what we did above; you only need to write one line of code._

In [ ]:
S = A[:3, :]
b_small = b[:3, :]
display(games.iloc[:3].drop(columns="game_id"))
P_small, L_small, U_small, y_small = plu_system(S, b_small)

# Your code goes below.
x_small = ...

print("Coefficients:")
print(x_small)

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include a screenshot of your completed code and its output.</p>

## Task 3: Compare with the full dataset (2 points)

---

Why are the first three rows enough to determine the same coefficient vector as the full dataset?

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include your explanation.</p>

## Task 4: Three special rows (5 points)

---

Above, we looked at the first three rows of the dataset. Here, we'll look at a different group of 3 rows: rows 5, 34, and 40. Run the cell below to extract just those 3 rows.

In [ ]:
dependent_rows = [5, 34, 40]
D = A[dependent_rows, :]
b_dependent = b[dependent_rows, :]
display(games.iloc[dependent_rows].drop(columns="game_id"))

Above, you should notice that the statistics in row 34 are all exactly the average of the corresponding statistics in rows 5 and 40. For example, the `FGM` in row 34 is $$\frac{26+36}{2} = 31.$$

Observe what happens if we perform a (P)LU factorization of the `FGM`, `3PM`, and `FTM` columns of the statistics dataset:

In [ ]:
D

In [ ]:
P_dep, L_dep, U_dep, y_dep = plu_system(D, b_dependent, display=True)

The matrix $U$ has a row of zeros, and $\vec y$'s last component is 0. This means that $U \vec x = \vec y$ is consistent, but there are infinitely many solutions for $\vec x$ in $U \vec x = \vec y$! 

Your job is to describe, in vector-parametric form, the complete set of solutions $\vec x$ in $U \vec x = \vec y$ above. Note that the decimals above are rounded to 4 decimal places; the $9.6111$ is really $9.6111111...$ and $12.1111$ is $12.1111111...$. To make the math simpler, note that
- $9.61111111... = 173/18$.
- $12.111111... = 218/18$.

Feel free to use a calculator (e.g., in a Python notebook cell).

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Write your solution set and show your work, including any screenshots of code you write to help you find the solutions.</p>

## Task 5: Look at the special games (1 point)

---

Using the shooting statistics and scores in the three rows above, explain why they are not sufficient to determine the scoring coefficients $x_1, x_2, x_3$ uniquely.

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include your explanation.</p>

<br><hr>

Now, let's forget about the special case of rows 5, 34, and 40. Let's return to considering the entire dataset of 660 rows. As we've established, the scoring rule

$$\text{score}=x_1(\text{FGM})+x_2(\text{3PM})+x_3(\text{FTM})$$

must be the same for every row of the dataset, meaning $x_1$, $x_2$, and $x_3$ must have a solution that works for every row in the dataset.

Let's suppose there was an error in our data recording process, in which one of the values in the vector $\vec b$ is corrupted and recorded incorrectly. Is there still a solution for $x_1$, $x_2$, and $x_3$? Let's explore.

The next cell deliberately introduces a 1 point error in one entry of $\vec b$. The shooting statistics (i.e. the values in the matrix $A$) and all other scores stay the same. Let's call the new vector $\vec b_{\text{changed}}$.

In [ ]:
changed_row = 137
b_changed = b.copy()
b_changed[changed_row, 0] += 1
display(games.iloc[[changed_row]].drop(columns="game_id"))
print("New score:", b_changed[changed_row, 0])

The question is, is there still a solution $\vec x$ for $A \vec x = \vec b_{\text{changed}}$, even though we've now changed one (and only one) of the 660 entries in $\vec b$ to get $\vec b_{\text{changed}}$?

To answer, we're going to introduce a new helper function that allows us to quickly check whether a solution exists: `solve`. Below, we define it and call it on `A` and `b_changed`.

In [ ]:
def solve(A, b):
    """
    Return one solution to Ax = b, if one exists.

    If there are infinitely many solutions, 
    return ONE of them.
    """
    x = np.linalg.lstsq(A, b, rcond=None)[0]
    if np.allclose(A @ x, b):
        return x
    else:
        raise ValueError('Ax = b has no solution.')

solve(A, b_changed)

## Task 6: Explain the effect of the error (2 points)

---

Above, we see that there is no $\vec x$ such that $A \vec x = \vec b_{\text{changed}}$: the `solve` function told us `Ax = b has no solution`. **The error is expected; you didn't do anything incorrectly!** 

Changing just one of the 660 values in $\vec b$ made it so that our previous solution of $\vec x = \begin{bmatrix} 2 \\ 1 \\ 1 \end{bmatrix}$ no longer works, and not only that, but there is no other solution.

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Explain why there is no solution to $A \vec x = \vec b_{\text{changed}}$.</p>

<br><hr>

Now, let's return to the original, unchanged dataset. Recall, the columns that we've stored in our $A$ have been `FGM` (field goals made), `3PM` (three-pointers made), and `FTM` (free throws made).

In [ ]:
A_with_col_names

So far, we haven't had a separate column for the number of two-pointers made. A two-pointer is any field goal that isn't a three-pointer. Let's experiment with what would happen if we did have a column storing the number of two-pointers, called `2PM`. Run the cell below to create a matrix $C$ that is $660 \times 4$: it has the same 3 columns as $A$, but also a new `2PM` column that appears last.

In [ ]:
games_extra = games.copy()
games_extra["2PM"] = games_extra["FGM"] - games_extra["3PM"]
C_with_col_names = games_extra[["FGM", "3PM", "FTM", "2PM"]]
C = C_with_col_names.to_numpy(dtype=float)
C_with_col_names

## Task 7: Solve the new system (1 point)

---

Use the function `solve` we defined earlier to solve for $\vec x$ in $C \vec x = \vec b$. This only requires one line of code.

In [ ]:
# Your code goes below.
x_for_four_columns = ...
x_for_four_columns

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include screenshots of your completed code and the output.</p>

## Task 8: Write the complete solution set in your report (5 points)

---

The result for `x_for_four_columns` you found above may not have been what you expected! A more natural solution, perhaps, would have been $$\vec x = \begin{bmatrix} 2 \\ 1 \\ 1 \\ 0 \end{bmatrix},$$ since the first three columns of $C$ are exactly the same as those in $A$, and we knew that $A \begin{bmatrix} 2 \\ 1 \\ 1 \end{bmatrix}$ was already equal to $\vec b$.

**It turns out that the system $C \vec x = \vec b$ has infinitely many solutions $\vec x$, even though $A \vec x = \vec b$ only had one solution for $\vec x$!** $\vec x = \begin{bmatrix} 2 \\ 1 \\ 1 \\ 0 \end{bmatrix}$ and your answer to Task 7 are two of these infinitely many solutions.

Your final task is to describe, in vector-parametric form, the complete solution set of $C \vec x = \vec b$.

_Hint: How is the new `2PM` column related to the existing columns of $C$? Use this relationship to identify a nonzero vector $\vec v$ satisfying $C \vec v = \vec 0$, and use that to find the complete set of $\vec x$'s where $C \vec x = \vec b$._

<p style="color: #e67e22;"><strong>In Your PDF:</strong> Include your complete solution set, and verify that the following three vectors belong to it.

<ol>
<li>$\vec x = \begin{bmatrix} 2 \\ 1 \\ 1 \\ 0 \end{bmatrix}$.</li>
<li>Your answer to Task 7. (Convert the decimals to fractions first; $1.666... = 5/3$, for example. The displayed values in Task 7 are rounded; every component can be written as an integer or fraction.)</li>
<li>$\vec x = \begin{bmatrix} 0 \\ 3 \\ 1 \\ 2 \end{bmatrix}$.</li>
</ol>

Which solution is the most intuitive to understand, in the context of basketball scoring?
</p>

## Finish line 🏁

Include the requested code, outputs, and written explanations for Tasks 1-8 in your homework PDF. You do not need to submit this notebook.